# Week 18, Day 4: Power, sample size and effect size

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w018/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Sample size for 80% power

Write `n_needed(d)` from the formula (round up). For d = 0.5, simulate 20,000 experiments of that size from N(0.5, 1), apply the two-sided z-test with σ = 1, and store the fraction significant in `power_sim`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2013)
z_a, z_b = 1.959964, 0.841621

def n_needed(d):
    return ...

n = n_needed(0.5)
xbar = rng.normal(0.5, 1, size=(20000, n)).mean(axis=1)
power_sim = ...
print(f"n for 80% power at d = 0.5: {n};  simulated power {power_sim:.3f};  at d = 0.2: {n_needed(0.2)}")

In [ ]:
import numpy as np
def test_formula():
    "n_needed rounds up (1.96 + 0.8416)^2 / d^2"
    assert n_needed(0.5) == int(np.ceil((1.959964 + 0.841621) ** 2 / 0.25))

def test_sim():
    "the simulated power at that sample size is within 0.03 of 0.8"
    assert abs(power_sim - 0.8) < 0.03

test_formula()
test_sim()
print("All checks passed")

### Exercise: How many trials to detect a response?

A neuron fires 5 spikes at baseline and 6 with the stimulus (Poisson), with no drift. For 10, 20, 40, 60, 80 and 120 trials, simulate 2,000 paired experiments each, test the mean difference with the z statistic mean / (sd / √n), and store the power for each in `powers`. Store the smallest trial count with power at least 0.8 in `trials_80`.

In [ ]:
from math import erf, sqrt
trial_counts = [10, 20, 40, 60, 80, 120]
powers = []
for n_tr in trial_counts:
    base = rng.poisson(5, (2000, n_tr)); resp = rng.poisson(6, (2000, n_tr))
    d = resp - base
    z = ...
    powers.append(...)
powers = np.array(powers)
trials_80 = ...
print("power by trials:", dict(zip(trial_counts, powers.round(3))), " trials for 80%:", trials_80)
plt.plot(trial_counts, powers, "o-"); plt.axhline(0.8, ls="--", c="k"); plt.xlabel("trials"); plt.ylabel("power"); plt.show()

In [ ]:
import numpy as np
def test_curve():
    "power rises with the number of trials"
    assert np.all(np.diff(powers) > -0.03) and powers[-1] > powers[0]

def test_target():
    "trials_80 is the smallest number of trials tried with power at least 0.8"
    assert powers[list(trial_counts).index(trials_80)] >= 0.8
    i = list(trial_counts).index(trials_80)
    assert i == 0 or powers[i - 1] < 0.8

test_curve()
test_target()
print("All checks passed")

### Exercise: Exaggerated effects

Simulate 20,000 studies of a true effect d = 0.3 with n = 20 (σ = 1). Keep only those significant by the two-sided z-test and compute `power_low`, the `exaggeration` ratio (mean |estimate| among significant studies / 0.3) and `sign_errors` (fraction of significant estimates below zero). Repeat the exaggeration ratio with n = 200 as `exaggeration_high`.

In [ ]:
def study(n, sims=20000):
    est = rng.normal(0.3, 1 / np.sqrt(n), sims)       # the estimated effect in each study
    sig = np.abs(est * np.sqrt(n)) > 1.96
    return est, sig

est, sig = study(20)
power_low = ...
exaggeration = ...
sign_errors = ...
est_h, sig_h = study(200)
exaggeration_high = ...
print(f"n = 20: power {power_low:.2f}, published effects {exaggeration:.2f} times too large, wrong sign {sign_errors:.3f};  n = 200: {exaggeration_high:.2f} times")

In [ ]:
import numpy as np
def test_low_power():
    "with n = 20 and d = 0.3, power is below 0.35"
    assert power_low < 0.35

def test_inflated():
    "significant estimates average more than 1.7 times the true effect"
    assert exaggeration > 1.7

def test_high_power():
    "with n = 200 the exaggeration ratio is below 1.15"
    assert exaggeration_high < 1.15

test_low_power()
test_inflated()
test_high_power()
print("All checks passed")